# 🪶 Qareed (قريض) v2 — Arabic Poetry Fine-Tuning (QLoRA on a 4-bit base)

**Task:** Arabic instruction -> 1-4 baits of Arabic poetry, optionally with a requested **theme**, **era** or **rhyme letter**.
**Base:** `Qwen/Qwen2.5-1.5B-Instruct` (switchable in 0.1). **Data:** `arbml/Ashaar_dataset`.

## What changed vs v1 (and why)
| Problem seen in v1 | v2 fix |
|---|---|
| Theme in the prompt was often ignored | Poem-level labels are noisy, so a window is used as "themed" only if **its own text** contains >= 2 theme keywords (weak labels, my heuristics: review them). 12 labels merged into 10 groups. |
| Rhyme was inconsistent | Every training window is **monorhyme** (same surface rhyme letter), and ~half of the prompts **ask for the rhyme letter** ("... على قافية حرف الراء"). Rhyme adherence is measurable from the data. |
| Loss included the constant system prompt | **Completion-only loss** (prompt/completion format) + a check that proves the mask before training starts. |
| Metrics were never run | Batched evaluation: BASE vs v2 (vs v1 if found on Drive): bait count, rhyme hit, rhyme consistency, theme hit, copy rate. |
| Data regenerated differently after a disconnect | Train/val files are saved to Drive and **reused** (`REBUILD_DATA=False`). Val is checked for bait-level leakage against train. |
| bf16 vs fp16 crash, OOM, silent merge failure | Auto fp16/bf16, LoRA in fp32, conservative batch, `expandable_segments`, **smoke test mode**, CPU merge, **verified upload**. |
| Pushed public by accident | `RUN_PUSH=False` by default, repos private by default. |

## Before you start
1. Runtime -> **T4 GPU**. 2. Colab Secret `HF_ACCESS_TOKEN` (Write permission), notebook access ON.
3. Recommended first run: `SMOKE_TEST=True` (30 steps, ~2 min) -> **restart runtime** -> set `False` -> run everything.
4. Training time is an **estimate**: v1 measured about 3.0 s/step on a T4 (1.5B model); the cell in 6.1 prints the projection for your step count.

## Honest expectations
Loss numbers are **not comparable with v1** (v1 averaged over the system prompt too). Meter (wazn) is still not controlled. "Rhyme" here is the **surface rhyme letter** (last letter, alef/maqsura skipped), not the full science of qafiya.

## 0) Setup

In [1]:
# 0.1 CONFIG — edit flags here only
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

BASE_MODEL   = "Qwen/Qwen2.5-1.5B-Instruct"   # 3B option: "Qwen/Qwen2.5-3B-Instruct" (slower; not tested here)
VERSION      = "v2"
RUN_NAME     = "Qareed-" + BASE_MODEL.split("/")[-1].replace("-Instruct", "") + f"-Poet-{VERSION}"

SMOKE_TEST   = False   # True: 30 steps only (memory/speed check). Then RESTART the runtime and set it back to False.
REBUILD_DATA = False   # False: reuse the saved train/val files from Drive when they exist
RUN_PUSH     = True   # True only when you are ready to publish (section 8)
PUSH_PRIVATE = True
PIN_VERSIONS = True    # the exact library versions that worked in v1
EPOCHS       = 1
MAX_TRAIN_SAMPLES = 30000
MAX_LEN      = 320
print("RUN_NAME:", RUN_NAME)

RUN_NAME: Qareed-Qwen2.5-1.5B-Poet-v2


In [2]:
# 0.2 install (restart the runtime if Colab asks, then continue from 0.3)
pins = "transformers==5.18.0 peft==0.21.2 trl==1.14.1 datasets==5.1.0" if PIN_VERSIONS else "transformers peft trl datasets"
!pip install -q -U {pins} accelerate bitsandbytes huggingface_hub pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 2.3.0 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [3]:
# 0.3 GPU + versions
import torch, transformers, peft, trl, datasets
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
USE_BF16 = torch.cuda.get_device_capability(0)[0] >= 8      # T4 (7.5) -> fp16 ; A100/L4 -> bf16
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
have = {"transformers": transformers.__version__, "peft": peft.__version__,
        "trl": trl.__version__, "datasets": datasets.__version__}
print(f"GPU: {GPU_NAME} | VRAM {VRAM_GB:.1f} GB | bf16: {USE_BF16} | torch {torch.__version__}")
print(have)
if PIN_VERSIONS:
    want = {"transformers": "5.18.0", "peft": "0.21.2", "trl": "1.14.1", "datasets": "5.1.0"}
    bad = {k: (have[k], v) for k, v in want.items() if have[k] != v}
    if bad:
        print("WARNING: version mismatch (have, want):", bad, "-> restart the runtime and re-run from 0.2")
if VRAM_GB < 14:
    print("WARNING: less than 14 GB of VRAM; expect to need a smaller batch (BS=4, ACC=4 in 6.1)")

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


GPU: Tesla T4 | VRAM 15.6 GB | bf16: False | torch 2.11.0+cu130
{'transformers': '5.18.0', 'peft': '0.21.2', 'trl': '1.14.1', 'datasets': '5.1.0'}


In [4]:
# 0.4 imports + Hugging Face login + Google Drive + paths + guard helper
import os, json, gc, re, math, random, collections, time, glob
import numpy as np
import pandas as pd
from google.colab import userdata, drive
from huggingface_hub import login, whoami, HfApi

try:
    token = userdata.get("HF_ACCESS_TOKEN")
except Exception:
    token = userdata.get("HF_TOKEN")
login(token=token)
HF_USER = whoami()["name"]
api = HfApi(token=token)
print("Logged in as:", HF_USER)

drive.mount("/content/drive")
WORK_DIR = "/content/drive/MyDrive/Colab Notebooks/Qareed"
os.makedirs(WORK_DIR, exist_ok=True)
OUT_DIR      = f"{WORK_DIR}/{RUN_NAME}" + ("-smoke" if SMOKE_TEST else "")
ADAPTER_DIR  = f"{OUT_DIR}/adapter"
TRAIN_JSONL  = f"{WORK_DIR}/qareed_{VERSION}_train.jsonl"
VAL_JSONL    = f"{WORK_DIR}/qareed_{VERSION}_val.jsonl"
ADAPTER_REPO = f"{HF_USER}/{RUN_NAME}-LoRA"
MERGED_REPO  = f"{HF_USER}/{RUN_NAME}"
MERGED_DIR   = f"/content/{RUN_NAME}-merged"          # local disk (not Drive): faster and safer

def require(*names, hint=""):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(f"Missing {missing}. {hint} (did the runtime restart? re-run the earlier sections)")

print("Work dir:", WORK_DIR, "| out dir:", OUT_DIR)

Logged in as: Youssefx64
Mounted at /content/drive
Work dir: /content/drive/MyDrive/Colab Notebooks/Qareed | out dir: /content/drive/MyDrive/Colab Notebooks/Qareed/Qareed-Qwen2.5-1.5B-Poet-v2


## 1) Data: load

Facts established in v1 (so we do not rediscover them):
- `poem_verses` is a numpy array of **shatrs** (hemistichs): two consecutive shatrs = one bait.
- `poem_meter` is a numeric id with no known name mapping -> **not used**.
- `poem_theme` is missing for ~71% of poems; `قصيره` and `عامه` are not real themes.
- The source text has **no `إ`** at all (dropped upstream).

In [ ]:
# 1.1 load the dataset (skipped when the saved v2 train/val files already exist on Drive)
DATA_READY = os.path.exists(TRAIN_JSONL) and os.path.exists(VAL_JSONL) and not REBUILD_DATA
print("DATA_READY (reuse saved files):", DATA_READY)

def shatrs_of(v):
    if isinstance(v, np.ndarray): v = v.tolist()
    if isinstance(v, (list, tuple)): return [str(x) for x in v]
    return str(v).split("\n")

if not DATA_READY:
    from datasets import load_dataset
    raw = load_dataset("arbml/Ashaar_dataset", split="train")
    df = raw.to_pandas()
    need = ["poem_verses", "poem_theme", "poet_era", "poem_language_type"]
    missing = [c for c in need if c not in df.columns]
    assert not missing, f"Dataset columns changed: missing {missing}; columns = {list(df.columns)}"
    print("Rows:", len(df))

DATA_READY (reuse saved files): True


## 2) Cleaning (shatrs -> baits)

In [ ]:
# 2.1 cleaning: pair shatrs, strip diacritics, patch the missing hamza, drop noisy/split-word baits
STRIP_DIACRITICS = True
BAIT_JOINER = "  ...  "

TATWEEL = "\u0640"
DIAC = re.compile(r"[\u064B-\u0652\u0670\u0653-\u065F]")
ARABIC_RE = re.compile(r"[\u0600-\u06FF]")
LATIN_RE = re.compile(r"[A-Za-z]")
DIGIT_RE = re.compile(r"[0-9\u0660-\u0669]")

# words that exist only because the source dropped the letter 'إ'
FIX = {
    "لى": "إلى", "ن": "إن", "ون": "وإن", "نما": "إنما", "ني": "إني", "نني": "إنني",
    "نه": "إنه", "نك": "إنك", "نها": "إنها", "نهم": "إنهم", "نهما": "إنهما", "نكم": "إنكم",
    "مام": "إمام", "براهيم": "إبراهيم", "نسان": "إنسان", "يمان": "إيمان", "حسان": "إحسان",
    "ذ": "إذ",
}
SHORT_OK = {"و","من","في","عن","لا","ما","يا","لم","لن","إن","أن","قد","بل","أو","ثم","هل",
            "كي","لو","إذ","ذو","مع","أم","أي","به","له","بي","لي","هو","هي","كم","ها","فا","وا","ال"}

def fix_missing_hamza(s): return " ".join(FIX.get(w, w) for w in s.split())

def clean_shatr(s):
    s = s.replace(TATWEEL, "")
    if STRIP_DIACRITICS: s = DIAC.sub("", s)
    s = re.sub(r"\s+", " ", s).strip()
    return fix_missing_hamza(s)

def shatr_ok(s):
    if not (6 <= len(s) <= 80):  return False
    if LATIN_RE.search(s):       return False
    if DIGIT_RE.search(s):       return False
    return len(ARABIC_RE.findall(s)) / max(1, len(s.replace(" ", ""))) >= 0.9

bait_stats = collections.Counter()

def bait_reason(a, b):
    if not (shatr_ok(a) and shatr_ok(b)):             return "noisy_shatr"
    ta, tb = a.split(), b.split()
    if "ذا" in ta or "ذا" in tb:                      return "ambiguous_idha"   # idha/hadha lost its hamza: unsafe to guess
    if len(ta[-1]) <= 2 and ta[-1] not in SHORT_OK:   return "split_word"
    if len(tb[0]) <= 2 and tb[0] not in SHORT_OK:     return "split_word"
    return None

def poem_to_baits(v):
    shatrs = [clean_shatr(x) for x in shatrs_of(v)]
    if len(shatrs) % 2 == 1: shatrs = shatrs[:-1]
    baits = []
    for i in range(0, len(shatrs), 2):
        a, b = shatrs[i], shatrs[i + 1]
        r = bait_reason(a, b)
        bait_stats[r or "ok"] += 1
        baits.append(None if r else f"{a}{BAIT_JOINER}{b}")
    return baits

def norm_label(x, strip=("قصيدة ", "شعر ")):
    if x is None or (isinstance(x, float) and pd.isna(x)): return None
    x = re.sub(r"\s+", " ", str(x)).strip()
    for p in strip:
        if x.startswith(p): x = x[len(p):]
    return None if x.lower() in {"", "none", "nan", "other", "غير محدد"} else x

if not DATA_READY:
    poems, dropped = [], collections.Counter()
    for v, th, era, lang in zip(df["poem_verses"], df["poem_theme"], df["poet_era"], df["poem_language_type"]):
        baits = poem_to_baits(v)
        good = [b for b in baits if b]
        if len(good) < 2:                 dropped["too_few_good_baits"] += 1; continue
        if len(good) / len(baits) < 0.7:  dropped["noisy_poem"] += 1; continue
        poems.append({"baits": good, "theme": norm_label(th), "era": norm_label(era), "lang": norm_label(lang)})
    print("Bait stats:", dict(bait_stats))
    print("Kept:", len(poems), "| Dropped:", dict(dropped))

    seen, uniq = set(), []
    for p in poems:
        key = " ".join(p["baits"][:2])
        if key not in seen: seen.add(key); uniq.append(p)
    poems = uniq
    print("After dedupe:", len(poems))

## 3) Instruction dataset (the part that matters most)

Design decisions:
- **Windows of 1-4 consecutive baits**, and only windows with a **consistent rhyme letter**.
- **Theme** is used only when the window text itself has >= `MIN_THEME_HITS` theme keywords. The keyword lists are *my heuristics*: read them and edit them.
- **Rhyme** is requested in about half of the prompts, always with the true letter taken from the answer.
- **Poem-level** train/val split, then a **bait-level leakage check**.

In [ ]:
# 3.1 tools: rhyme key, theme groups + keyword regexes, prompt templates
# ---- rhyme (surface rhyme letter) ----
LETTER_NAME = {
    "ء": "الهمزة", "ب": "الباء", "ت": "التاء", "ث": "الثاء", "ج": "الجيم", "ح": "الحاء", "خ": "الخاء",
    "د": "الدال", "ذ": "الذال", "ر": "الراء", "ز": "الزاي", "س": "السين", "ش": "الشين", "ص": "الصاد",
    "ض": "الضاد", "ط": "الطاء", "ظ": "الظاء", "ع": "العين", "غ": "الغين", "ف": "الفاء", "ق": "القاف",
    "ك": "الكاف", "ل": "اللام", "م": "الميم", "ن": "النون", "ه": "الهاء", "و": "الواو", "ي": "الياء",
}
R_NORM = str.maketrans({"ى": "ا", "أ": "ا", "إ": "ا", "آ": "ا", "ؤ": "و", "ئ": "ي", "ة": "ه"})

def last_word(line):
    last = re.split(r"\.\.\.|…", line)[-1]
    words = DIAC.sub("", last).split()
    return re.sub(r"[^\u0621-\u064A]", "", words[-1]) if words else ""

def rhyme_key(line):
    """Surface rhyme letter of a bait (last letter; a final alef/maqsura is a vowel extension, so skip it)."""
    w = last_word(line).translate(R_NORM)
    if not w: return None
    if w[-1] == "ا" and len(w) >= 2:
        w = w[:-1]
    k = w[-1]
    return k if k in LETTER_NAME else None

def window_rhyme(window):
    keys = [rhyme_key(b) for b in window]
    return keys[0] if keys and keys[0] and all(k == keys[0] for k in keys) else None

# ---- theme groups (weak labels: poem-level label AND keywords in the window itself) ----
THEME_GROUPS = {
    "الغزل والحب": dict(src={"غزل", "رومنسيه"}, phrases=["الغزل", "الحب والهيام", "الغزل بالمحبوبة"],
        kw=["حبيب","حبي","الحب","عشق","غرام","هوى","وجنة","خدود","ثغر","لحظ","جفن","جفون","قوام","غزال","ريم","ظبي","فتنة","محبوب","عاشق","هيام","فؤاد"]),
    "الحزن": dict(src={"حزينه"}, phrases=["الحزن", "الحزن والأسى"],
        kw=["حزن","دمع","دموع","بكى","بكا","بكاء","يبكي","تبكي","أبكي","أسى","ألم","لوعة","جوى","نحيب","أنين","شجن","وجع","يأس","كئيب","عبرة","أحزان","مصيبة"]),
    "الرثاء": dict(src={"رثاء"}, phrases=["الرثاء", "رثاء الفقيد"],
        kw=["رثا","رثاء","فقيد","المنون","منية","قبر","الردى","ردى","نعي","ضريح","ثكل","موت","مات","الحمام","رحل","الفناء","لحد","الثرى","نعش"]),
    "الشوق والحنين": dict(src={"شوق"}, phrases=["الشوق", "الشوق والحنين"],
        kw=["شوق","حنين","اشتياق","أشتاق","لهف","لقاء","ذكرى","تذكر","سهاد","وجد","أشواق","اشواق"]),
    "الفراق": dict(src={"فراق"}, phrases=["الفراق", "البعد والفراق"],
        kw=["فراق","البين","وداع","رحيل","النوى","هجر","ودعت","ودعني","بعدك","غيابك","بعدي","ارتحل","رحلوا","ظعن","ظعائن","غياب","بعاد"]),
    "المدح": dict(src={"مدح"}, phrases=["المدح", "الثناء والمدح"],
        kw=["كريم","جود","مجد","شرف","أمير","فضل","همة","سماحة","الندى","سخاء","المعالي","الكرام","ثناء","عظيم","ممدوح","مدحي","مدح","سيد","ملك","كرم"]),
    "العتاب": dict(src={"عتاب"}, phrases=["العتاب", "العتاب واللوم"],
        kw=["عتب","عتاب","لوم","جفوة","ظلم","خلان","صديق","وفاء","غدر","قسوة","أخلف","جفا","جفاء","تلوم","عذل","عاذل"]),
    "الهجاء": dict(src={"هجاء", "ذم"}, phrases=["الهجاء", "الهجاء والتهكم", "الذم"],
        kw=["لئيم","جبان","بخل","بخيل","خسيس","وضيع","ذليل","دنيء","نذل","حقير","جهل","كلب","حمار","خبيث","سفيه","فاجر","لؤم","خسة","هجاء","هجوت","وقح","الدنيا","غدر","خيانة","ظالم","فاسد"]),
    "الدين والتقوى": dict(src={"دينية"}, phrases=["الدين والتقوى", "الإيمان والدعاء"],
        kw=["الله","ربي","دعاء","النبي","محمد","رسول","صلاة","الجنة","النار","ذنب","توبة","إيمان","الدين","استغفر","رحمة","اللهم","سبحان","تقوى","عفو","غفران","الرحمن","ربنا"]),
    "حب الوطن": dict(src={"وطنيه"}, phrases=["حب الوطن", "الوطنية"],
        kw=["وطن","الوطن","بلاد","أرض","شعب","الأمة","العرب","حرية","شهيد","مصر","فلسطين","القدس","عروبة","نضال","الجهاد","أوطان","الأوطان","الديار","الحمى","استقلال","تراب","الشام","بغداد","لبنان"]),
}
SRC2GROUP = {s: g for g, d in THEME_GROUPS.items() for s in d["src"]}

def build_kw_regex(kws):
    kws = sorted({k for k in kws if len(k) >= 3}, key=len, reverse=True)
    alt = "|".join(re.escape(k) for k in kws)
    return re.compile(r"(?<![\u0621-\u064A])(?:وال|فال|بال|كال|لل|ال|و|ف|ب|ل|ك)?(" + alt + r")[\u0621-\u064A]{0,3}(?![\u0621-\u064A])")

for g, d in THEME_GROUPS.items():
    d["rx"] = build_kw_regex(d["kw"])

def kw_hits(text, group):
    return len({m.group(1) for m in THEME_GROUPS[group]["rx"].finditer(text)})

# ---- prompts ----
SYSTEM_PROMPT = ("أنت «قريض»، شاعر عربي متمكّن من اللغة والعروض والقافية. "
                 "تكتب شعرًا عربيًا فصيحًا، وتلتزم بعدد الأبيات وبالقافية المطلوبة إن وُجدت، دون شرح أو تعليق.")
V1_SYSTEM_PROMPT = ("أنت «قريض»، شاعر عربي متمكّن من اللغة والعروض والقافية. "
                    "تكتب شعرًا عربيًا فصيحًا موزونًا ومقفّى، وتلتزم بعدد الأبيات المطلوب دون شرح أو تعليق.")
NUM_WORDS = {1: "بيتًا واحدًا", 2: "بيتين", 3: "ثلاثة أبيات", 4: "أربعة أبيات"}
ERA_MAP = {"قبل الإسلام": "ما قبل الإسلام", "عصر بين الدولتين": None}

T_THEME     = ["اكتب {n} من الشعر في {theme}", "قل لي {n} عن {theme}", "أريد {n} من الشعر العربي في موضوع {theme}",
               "انظم {n} في {theme}", "هل تستطيع أن تكتب لي {n} عن {theme}"]
T_THEME_ERA = ["اكتب {n} في {theme} بأسلوب شعراء {era}", "انظم {n} في {theme} على طريقة شعراء {era}"]
T_ERA_ONLY  = ["اكتب {n} من الشعر بأسلوب شعراء {era}", "انظم {n} على طريقة شعراء {era}"]
T_PLAIN     = ["اكتب {n} من الشعر العربي", "قل لي {n} من الشعر", "انظم {n} من الشعر"]
RHYME_SUFFIX = [(" على قافية حرف {L}", ""), ("، والروي حرف {L}", "")]

def make_prompt(n, group, era, rhyme):
    n_txt = NUM_WORDS[n]
    era_txt = ERA_MAP.get(era, era) if era else None
    if group:
        theme = random.choice(THEME_GROUPS[group]["phrases"])
        if era_txt: core = random.choice(T_THEME_ERA).format(n=n_txt, theme=theme, era=era_txt)
        else:       core = random.choice(T_THEME).format(n=n_txt, theme=theme)
    elif era_txt:   core = random.choice(T_ERA_ONLY).format(n=n_txt, era=era_txt)
    else:           core = random.choice(T_PLAIN).format(n=n_txt)
    if rhyme:
        suffix, _ = random.choice(RHYME_SUFFIX)
        core += suffix.format(L=LETTER_NAME[rhyme])
    return core + ("؟" if core.startswith("هل") else ".")

random.seed(0)
for args_ in [(2, "الحزن", None, None), (2, "المدح", "العصر العباسي", None), (3, "الشوق والحنين", None, "ر"),
              (2, None, None, "ل"), (1, None, "العصر المملوكي", None), (2, None, None, None)]:
    print(make_prompt(*args_))
print("rhyme_key tests:", rhyme_key("قلبي يحن إلى الديار  ...  وأشتاق للبحر الكبير"), rhyme_key("وما ذاك إلا  ...  هدى"),
      "| kw_hits:", kw_hits("وأبكي دموعي على فراقك وحبيبي بعيد", "الحزن"))

انظم بيتين في الحزن والأسى.
انظم بيتين في المدح على طريقة شعراء العصر العباسي.
انظم ثلاثة أبيات في الشوق والحنين، والروي حرف الراء.
قل لي بيتين من الشعر، والروي حرف اللام.
اكتب بيتًا واحدًا من الشعر بأسلوب شعراء العصر المملوكي.
انظم بيتين من الشعر.
rhyme_key tests: ر د | kw_hits: 2


In [ ]:
# 3.2 sampling logic
MIN_THEME_HITS = 2          # distinct theme keywords required inside the window
P_RHYME_THEMED, P_ERA_THEMED = 0.40, 0.15
P_RHYME_OTHER,  P_ERA_OTHER  = 0.60, 0.40     # P_ERA_OTHER is conditional on "no rhyme asked"
N_CHOICES = [1, 2, 2, 2, 3, 4]
P_START_AT_OPENING = 0.6    # opening baits express the poem theme best
MAX_BAITS_THEMED = 16
WINDOW_TRIES = 12

def pick_window(baits, opening_bias):
    n = min(random.choice(N_CHOICES), len(baits))
    start = 0 if (opening_bias and random.random() < P_START_AT_OPENING) else random.randint(0, len(baits) - n)
    return n, baits[start:start + n]

def try_window(poem, group):
    for _ in range(WINDOW_TRIES):
        n, window = pick_window(poem["baits"], opening_bias=bool(group))
        key = window_rhyme(window)
        if key is None: continue
        if group and kw_hits(" ".join(window), group) < MIN_THEME_HITS: continue
        return n, window, key
    return None

def build_samples(poem, k, group=None):
    out = []
    for _ in range(k):
        r = try_window(poem, group)
        if r is None: continue
        n, window, key = r
        era_ok = bool(poem["era"]) and bool(ERA_MAP.get(poem["era"], poem["era"]))
        if group:
            want_rhyme = random.random() < P_RHYME_THEMED
            use_era = (not want_rhyme) and era_ok and random.random() < P_ERA_THEMED
        else:
            want_rhyme = random.random() < P_RHYME_OTHER
            use_era = (not want_rhyme) and era_ok and random.random() < P_ERA_OTHER
        era = poem["era"] if use_era else None
        out.append({"prompt": make_prompt(n, group, era, key if want_rhyme else None),
                    "answer": "\n".join(window), "n_baits": n, "theme": group,
                    "rhyme": key if want_rhyme else None, "era": era})
    return out

In [ ]:
# 3.3 build (or load) the train/val samples
def save_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

if DATA_READY:
    train_samples, val_samples = load_jsonl(TRAIN_JSONL), load_jsonl(VAL_JSONL)
    print("Loaded saved data:", len(train_samples), len(val_samples))
else:
    random.seed(42)
    BAD_LANG = {"عامي", "شعبي", "-"}                      # colloquial / folk poems: we want a Fusha model
    poems_f = [p for p in poems if p.get("lang") not in BAD_LANG]
    random.shuffle(poems_f)
    for p in poems_f:
        p["group"] = SRC2GROUP.get(p["theme"])
    themed = [p for p in poems_f if p["group"] and len(p["baits"]) <= MAX_BAITS_THEMED]
    other  = [p for p in poems_f if not (p["group"] and len(p["baits"]) <= MAX_BAITS_THEMED)]
    print("poems after language filter:", len(poems_f), "| themed:", len(themed), "| other:", len(other))

    # poem-level split FIRST (a poem never appears in both)
    VAL_THEMED_POEMS, VAL_OTHER_POEMS = 350, 450
    val_poems = themed[:VAL_THEMED_POEMS] + other[:VAL_OTHER_POEMS]
    themed_tr, other_tr = themed[VAL_THEMED_POEMS:], other[VAL_OTHER_POEMS:]

    MAX_POEMS_PER_GROUP, OTHER_TRAIN_POEMS = 2500, 14000
    by_group = collections.defaultdict(list)
    for p in themed_tr: by_group[p["group"]].append(p)
    themed_tr = [p for lst in by_group.values() for p in lst[:MAX_POEMS_PER_GROUP]]
    other_tr = other_tr[:OTHER_TRAIN_POEMS]

    train_samples  = [s for p in themed_tr for s in build_samples(p, 3, group=p["group"])]
    train_samples += [s for p in other_tr  for s in build_samples(p, 1, group=None)]
    random.shuffle(train_samples)
    print("train samples (before cap):", len(train_samples))
    train_samples = train_samples[:MAX_TRAIN_SAMPLES]

    val_samples  = [s for p in val_poems if p["group"] and len(p["baits"]) <= MAX_BAITS_THEMED for s in build_samples(p, 1, group=p["group"])]
    val_samples += [s for p in val_poems if not (p["group"] and len(p["baits"]) <= MAX_BAITS_THEMED) for s in build_samples(p, 1, group=None)]
    random.shuffle(val_samples)

    # bait-level leakage check: drop val samples that share any bait with train
    train_baits = {b for s in train_samples for b in s["answer"].split("\n")}
    before = len(val_samples)
    val_samples = [s for s in val_samples if not any(b in train_baits for b in s["answer"].split("\n"))]
    print(f"val leakage check: removed {before - len(val_samples)} of {before}")

    save_jsonl(TRAIN_JSONL, train_samples); save_jsonl(VAL_JSONL, val_samples)
    print("saved to Drive:", TRAIN_JSONL, VAL_JSONL)
print("train:", len(train_samples), "| val:", len(val_samples))

Loaded saved data: 17904 505
train: 17904 | val: 505


In [ ]:
# 3.4 what is in the data? (read this before training)
def describe(samples, title):
    d = pd.DataFrame(samples)
    print(f"\n=== {title}: {len(d)} samples ===")
    print("n_baits:", d.n_baits.value_counts().sort_index().to_dict())
    print("with theme:", int(d.theme.notna().sum()), "| with rhyme:", int(d.rhyme.notna().sum()), "| with era:", int(d.era.notna().sum()))
    print(d.theme.fillna("— none —").value_counts().to_string())
    print("top rhyme letters asked:", d.rhyme.dropna().value_counts().head(10).to_dict())
    return d

dtr = describe(train_samples, "TRAIN"); dva = describe(val_samples, "VAL")
weak = [g for g in THEME_GROUPS if (dtr.theme == g).sum() < 300]
if weak: print("\nNOTE: few training samples for:", weak, "(expect weaker control on them)")
if len(val_samples) < 200: print("WARNING: very small validation set")

random.seed(7)
pool = [s for s in train_samples if s["theme"]][:2000]
for s in random.sample(pool, min(4, len(pool))) + random.sample([s for s in train_samples if s["rhyme"]], 4):
    print("-" * 60); print("Q:", s["prompt"]); print("A:\n" + s["answer"])


=== TRAIN: 17904 samples ===
n_baits: {1: 3711, 2: 9053, 3: 2704, 4: 2436}
with theme: 3979 | with rhyme: 9986 | with era: 1738
theme
— none —         13925
الغزل والحب       1509
المدح              996
الحزن              431
الدين والتقوى      395
الشوق والحنين      138
الفراق             123
العتاب             123
الرثاء             105
الهجاء              83
حب الوطن            76
top rhyme letters asked: {'ه': 1344, 'ر': 1211, 'م': 883, 'ل': 840, 'د': 838, 'ب': 826, 'ي': 784, 'ن': 784, 'ق': 343, 'ع': 307}

=== VAL: 505 samples ===
n_baits: {1: 98, 2: 284, 3: 60, 4: 63}
with theme: 59 | with rhyme: 267 | with era: 56
theme
— none —         446
الغزل والحب       23
المدح             20
الدين والتقوى      7
الهجاء             3
الفراق             2
الحزن              2
الرثاء             2
top rhyme letters asked: {'ر': 41, 'ب': 28, 'ه': 28, 'ي': 25, 'ن': 23, 'م': 19, 'د': 16, 'ل': 16, 'ع': 14, 'ق': 11}

NOTE: few training samples for: ['الرثاء', 'الشوق والحنين', 'الفراق', 'العتاب', 

## 4) Training format: prompt/completion (loss on the poet's answer only)

In [ ]:
# 4.1 tokenizer + prompt/completion datasets + length check
require("train_samples", "val_samples", "SYSTEM_PROMPT", hint="Run section 3 first.")
from transformers import AutoTokenizer
from datasets import Dataset

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def to_pc(s):
    return {"prompt": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": s["prompt"]}],
            "completion": [{"role": "assistant", "content": s["answer"]}]}

train_ds = Dataset.from_list([to_pc(s) for s in train_samples])
val_ds   = Dataset.from_list([to_pc(s) for s in val_samples])

def full_text(ex):
    return tok.apply_chat_template(ex["prompt"] + ex["completion"], tokenize=False)

probe = train_ds.select(range(min(3000, len(train_ds))))
lens = [len(tok(full_text(e)).input_ids) for e in probe]
print("Token length — mean:", int(np.mean(lens)), "| p95:", int(np.percentile(lens, 95)), "| p99:", int(np.percentile(lens, 99)), "| max:", max(lens))
over = sum(l > MAX_LEN for l in lens)
print(f"samples longer than MAX_LEN={MAX_LEN}: {over}/{len(lens)}", "(raise MAX_LEN if > 1%)" if over / len(lens) > 0.01 else "(fine)")
print(full_text(train_ds[0]))

Token length — mean: 149 | p95: 201 | p99: 211 | max: 231
samples longer than MAX_LEN=320: 0/3000 (fine)
<|im_start|>system
أنت «قريض»، شاعر عربي متمكّن من اللغة والعروض والقافية. تكتب شعرًا عربيًا فصيحًا، وتلتزم بعدد الأبيات وبالقافية المطلوبة إن وُجدت، دون شرح أو تعليق.<|im_end|>
<|im_start|>user
انظم بيتين من الشعر، والروي حرف السين.<|im_end|>
<|im_start|>assistant
دارت دوائره للأولياء ولل  ...  أعداء ما بين سعاد وتعاس
للعقل أدهش إذ ما زال مبتسما  ...  مع انه أسد من ل عباس<|im_end|>



## 5) Model + baseline

If a trainer/model already exists in this session (for example after a smoke test), **restart the runtime** instead of reloading on top of it: a second `SFTTrainer` on the same model stacks adapters.

In [ ]:
# 5.1 cleanup (safe on a fresh session) + load the base model in 4-bit
for name in ["trainer", "model"]:
    if name in globals():
        del globals()[name]
gc.collect(); torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto", dtype=COMPUTE_DTYPE)
model.config.use_cache = False
print("Loaded. GPU memory (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded. GPU memory (GB): 1.14


In [ ]:
# 5.2 generation helpers + baseline (before training)
@torch.no_grad()
def generate(prompt, mdl, system_prompt=None, max_new_tokens=160, temperature=0.8, top_p=0.9, rep_pen=1.1):
    msgs = [{"role": "system", "content": system_prompt or SYSTEM_PROMPT}, {"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tok(text, return_tensors="pt").to(mdl.device)
    mdl.eval()
    out = mdl.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature,
                       top_p=top_p, repetition_penalty=rep_pen, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True).strip()

TEST_PROMPTS = [
    "اكتب بيتين من الشعر في الحزن.",
    "اكتب بيتين على قافية حرف الراء.",
    "انظم ثلاثة أبيات في الشوق والحنين على قافية حرف اللام.",
    "أريد بيتين من الشعر العربي في الغزل.",
    "اكتب بيتًا واحدًا عن الصبر.",                    # theme the model never saw as a label
    "اكتب أربعة أبيات في رثاء الأم على قافية حرف الميم.",   # unseen phrasing
    "اكتب بيتين من الشعر بأسلوب شعراء العصر العباسي.",
    "انظم بيتين في المدح على قافية حرف الدال.",
]

RUN_BASELINE = True
if RUN_BASELINE:
    print("===== BASELINE (before fine-tuning) =====")
    for p in TEST_PROMPTS:
        print("\nQ:", p); print(generate(p, model))

===== BASELINE (before fine-tuning) =====

Q: اكتب بيتين من الشعر في الحزن.
عذراً، كم أنصحك بأن أكتب هذه الكلمات؟ أنا هنا لمساعدتك وليس لتوجيهك إلى أي موضوع. يمكنك كتابة ما تريد أن تحكيه بكل سهولة وحرية. أنت قوي وفريد ومتميز. هل هناك شيء معين يجذبك؟ استمع للإشارات التي تأتي لك من القلب.

Q: اكتب بيتين على قافية حرف الراء.
عندما ينطلق الصوت
فإن أثره يتلاشى في الليل.

Q: انظم ثلاثة أبيات في الشوق والحنين على قافية حرف اللام.
وما كان لي من ريب
لما كنت أستمع لساجدي
لم يخف القلب العمياءً 
بجواري كل يومين

Q: أريد بيتين من الشعر العربي في الغزل.
أنا آسف، ولكن ليس لدي القدرة على كتابة الشعر العربي بشكل مباشر. يمكنني التعبير عن الرغبات والمشاعر بطريقة أخرى، مثل النصائح أو الأقوال الشهيرة، لكن الشعر الفصيح الذي يحتوي على أبيات معينة من القافية العربية ليس ضمن قدرتي. يمكنك أن تكتب الشعر في مكان آخر أو استخدام أدوات الذكاء الصناعي لكتابة الشعر للمساعدة في إنتاج الشعر العربي. هل هناك شيء آخر تحتاج إلى مساعدتك فيه؟

Q: اكتب بيتًا واحدًا عن الصبر.
في صمت الظلام، أخذت القهوة طلقة، ثم بدأت الأحلام تنا

## 6) Training

- Batch **8 x accumulation 2** with gradient checkpointing: the configuration verified on a T4 in v1 (about 2 GB peak). Do not raise the batch on a T4 (a bigger batch ran out of memory in v1 because of the large vocabulary logits).
- Smoke test first (`SMOKE_TEST=True`): confirms memory, speed and the label mask, then **restart the runtime**.

In [ ]:
# 6.1 LoRA + trainer config
require("train_ds", "val_ds", "model", "tok", hint="Run sections 4-5 first.")
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

peft_cfg = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

BS, ACC = 8, 2                      # if you ever get CUDA OOM: BS, ACC = 4, 4
steps_per_epoch = math.ceil(len(train_ds) / (BS * ACC))
total_steps = steps_per_epoch * EPOCHS
SEC_PER_STEP = 3.0                  # measured in v1 on a T4 with the 1.5B model (about 2970 s / 1250 steps)
print(f"steps: {total_steps} | rough time estimate: {total_steps * SEC_PER_STEP / 60:.0f} min "
      f"(extrapolated from v1; the 3B model would be slower)")

cfg = dict(
    output_dir=OUT_DIR, num_train_epochs=EPOCHS, learning_rate=2e-4, lr_scheduler_type="cosine",
    warmup_steps=60, weight_decay=0.01, max_grad_norm=0.3,
    fp16=not USE_BF16, bf16=USE_BF16, optim="paged_adamw_8bit",
    per_device_train_batch_size=BS, per_device_eval_batch_size=BS, gradient_accumulation_steps=ACC,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=25, eval_strategy="steps", eval_steps=250,
    save_strategy="steps", save_steps=250, save_total_limit=3,
    report_to="none", max_length=MAX_LEN, packing=False, seed=42,
)
if SMOKE_TEST:
    cfg.update(max_steps=30, eval_strategy="no", save_strategy="no", logging_steps=5)

try:
    args = SFTConfig(**cfg, completion_only_loss=True)
except TypeError as e:
    print("completion_only_loss not accepted by this TRL version; relying on the default for prompt/completion data:", e)
    args = SFTConfig(**cfg)

trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                     processing_class=tok, peft_config=peft_cfg)
trainer.model.print_trainable_parameters()

steps: 1119 | rough time estimate: 56 min (extrapolated from v1; the 3B model would be slower)


Tokenizing train dataset:   0%|          | 0/17904 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/17904 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/17904 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/17904 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/505 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/505 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/505 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/505 [00:00<?, ? examples/s]

trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364


In [ ]:
# 6.2 safety checks BEFORE spending an hour: (a) LoRA weights fp32, (b) the loss mask covers only the answer
require("trainer")
for p in trainer.model.parameters():
    if p.requires_grad:
        p.data = p.data.float()           # fp16 GradScaler cannot handle bf16 grads (v1 crash) -> keep LoRA in fp32
dt = collections.Counter(str(p.dtype) for p in trainer.model.parameters() if p.requires_grad)
print("trainable dtypes:", dict(dt)); assert set(dt) == {"torch.float32"}, dt

ex = trainer.train_dataset[0]
print("tokenized columns:", list(ex.keys()))
ids = ex["input_ids"]
if "labels" in ex:
    sel = [i for i, l in zip(ids, ex["labels"]) if l != -100]
elif "completion_mask" in ex:
    sel = [i for i, m in zip(ids, ex["completion_mask"]) if m == 1]
else:
    sel = None
if sel is None:
    print("WARNING: cannot verify the loss mask with this TRL version. Check the first logged loss: it should look higher than v1's (about 2+ instead of about 1.4).")
else:
    dec = tok.decode(sel)
    print("TOKENS THAT COUNT IN THE LOSS:\n", dec)
    if SYSTEM_PROMPT[:15] in dec or len(sel) >= len(ids) - 3:
        raise RuntimeError("The loss still covers the prompt. Do not train; send me this output.")
    print("OK: the loss covers only the assistant answer.")

trainable dtypes: {'torch.float32': 392}
tokenized columns: ['prompt', 'completion', 'input_ids', 'labels']
TOKENS THAT COUNT IN THE LOSS:
 دارت دوائره للأولياء ولل  ...  أعداء ما بين سعاد وتعاس
للعقل أدهش إذ ما زال مبتسما  ...  مع انه أسد من ل عباس<|im_end|>

OK: the loss covers only the assistant answer.


In [ ]:
# 6.3 train (resumes from the last checkpoint on Drive after a disconnect)
require("trainer")
ckpts = [] if SMOKE_TEST else sorted(glob.glob(f"{OUT_DIR}/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
print("resuming from:", ckpts[-1] if ckpts else "scratch")
t0 = time.time()
trainer.train(resume_from_checkpoint=ckpts[-1] if ckpts else None)
elapsed_min = (time.time() - t0) / 60
print("Elapsed (min):", round(elapsed_min, 1), "| Peak GPU mem (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 2))
evals = [(h["step"], h["eval_loss"]) for h in trainer.state.log_history if "eval_loss" in h]
print("eval_loss history:", evals)

resuming from: scratch


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
250,3.497464,3.572966,3.582309,599028.000000,0.324399
500,3.407002,3.482530,3.444451,1198105.000000,0.333532
750,3.332313,3.426517,3.411503,1794953.000000,0.343980
1000,3.300976,3.398874,3.407250,2392633.000000,0.347888
1119,3.281639,3.397276,3.408040,2676088.000000,0.348900


Elapsed (min): 52.7 | Peak GPU mem (GB): 2.37
eval_loss history: [(250, 3.5729658603668213), (500, 3.482530117034912), (750, 3.4265174865722656), (1000, 3.398874044418335), (1119, 3.3972764015197754)]


In [ ]:
# 6.4 save the adapter + a run record (everything needed to reproduce)
require("trainer")
trainer.model.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
record = dict(run=RUN_NAME, system_prompt=SYSTEM_PROMPT, base_model=BASE_MODEL, versions=have, gpu=GPU_NAME, smoke=SMOKE_TEST,
              n_train=len(train_samples), n_val=len(val_samples), epochs=EPOCHS, bs=BS, acc=ACC, max_len=MAX_LEN,
              lora=dict(r=32, alpha=64), lr=2e-4, min_theme_hits=MIN_THEME_HITS,
              eval_loss_history=[(h["step"], h["eval_loss"]) for h in trainer.state.log_history if "eval_loss" in h])
with open(f"{OUT_DIR}/run_record.json", "w", encoding="utf-8") as f:
    json.dump(record, f, ensure_ascii=False, indent=1)
print("Saved adapter to", ADAPTER_DIR)
if SMOKE_TEST:
    print("\nSMOKE TEST DONE: memory/speed/mask are fine. Now RESTART the runtime, set SMOKE_TEST=False in 0.1 and run again.")

Saved adapter to /content/drive/MyDrive/Colab Notebooks/Qareed/Qareed-Qwen2.5-1.5B-Poet-v2/adapter


## 7) Evaluation: BASE vs v2 (vs v1 when found on Drive)

All metrics are **automatic heuristics on held-out poems**, not human judgement; read the printed examples yourself.

| metric | meaning |
|---|---|
| `count_match` | produced exactly the requested number of baits |
| `rhyme_hit` | (rhyme prompts) every bait ends with the requested letter |
| `rhyme_consistency` | share of baits ending with the majority letter (needs >= 2 baits) |
| `theme_hit` | (theme prompts) the output contains >= 1 keyword of the theme (the same heuristic lexicon used for the training filter) |
| `copy_rate` | share of generated baits that appear verbatim in the training set (memorization) |
| `no_latin` | no Latin letters (no code-switching) |

In [ ]:
# 7.1 metric helpers + batched generation
require("trainer", "val_samples", "train_samples", hint="Train in this session first (or reload the adapter).")
from contextlib import nullcontext

tok.padding_side = "left"                      # needed for batched generation
trainer.model.config.use_cache = True
norm_bait = lambda b: re.sub(r"[\s.…]+", "", b)
train_bait_norm = {norm_bait(b) for s in train_samples for b in s["answer"].split("\n")}

def parse_baits(text): return [l.strip() for l in text.split("\n") if l.strip()]

def one_metrics(s, out):
    baits = parse_baits(out)
    keys = [rhyme_key(b) for b in baits]
    valid = [k for k in keys if k]
    cons = collections.Counter(valid).most_common(1)[0][1] / len(valid) if len(valid) >= 2 else None
    hit = (len(baits) > 0 and len(valid) == len(baits) and all(k == s["rhyme"] for k in valid)) if s["rhyme"] else None
    th = (kw_hits(out, s["theme"]) >= 1) if s["theme"] else None
    copy = (sum(norm_bait(b) in train_bait_norm for b in baits) / len(baits)) if baits else 0.0
    return dict(count_match=len(baits) == s["n_baits"], no_latin=not LATIN_RE.search(out),
                rhyme_consistency=cons, rhyme_hit=hit, theme_hit=th, copy_rate=copy)

def aggregate(samples, outs):
    d = pd.DataFrame([one_metrics(s, o) for s, o in zip(samples, outs)])
    res = {c: (None if d[c].dropna().empty else round(float(d[c].dropna().astype(float).mean()), 3)) for c in d.columns}
    res["n"] = len(d)
    return res

@torch.no_grad()
def gen_many(prompts, mdl, system_prompt=None, bs=10, max_new_tokens=160, temperature=0.7, seed=1234):
    system_prompt = system_prompt or SYSTEM_PROMPT
    outs = []
    mdl.eval()
    for i in range(0, len(prompts), bs):
        texts = [tok.apply_chat_template([{"role": "system", "content": system_prompt}, {"role": "user", "content": p}],
                                         tokenize=False, add_generation_prompt=True) for p in prompts[i:i + bs]]
        enc = tok(texts, return_tensors="pt", padding=True).to(mdl.device)
        torch.manual_seed(seed + i)
        out = mdl.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature, top_p=0.9,
                           repetition_penalty=1.1, pad_token_id=tok.pad_token_id)
        outs += [tok.decode(r[enc.input_ids.shape[1]:], skip_special_tokens=True).strip() for r in out]
    return outs

EVAL_N = 60
EVAL_SETS = {
    "rhyme": [s for s in val_samples if s["rhyme"]][:EVAL_N],
    "theme": [s for s in val_samples if s["theme"] and not s["rhyme"]][:EVAL_N],
    "plain": [s for s in val_samples if not s["theme"] and not s["rhyme"]][:EVAL_N],
}
print({k: len(v) for k, v in EVAL_SETS.items()})

{'rhyme': 60, 'theme': 40, 'plain': 60}


In [ ]:
# 7.2 run BASE vs TUNED (adapter on/off) — takes a while; batched
RESULTS, SAVED_OUTS = {}, {}
for variant, ctx in [("BASE", trainer.model.disable_adapter()), ("TUNED", nullcontext())]:
    RESULTS[variant], SAVED_OUTS[variant] = {}, {}
    with ctx:
        for grp, smp in EVAL_SETS.items():
            if not smp: continue
            outs = gen_many([s["prompt"] for s in smp], trainer.model)
            SAVED_OUTS[variant][grp] = outs
            RESULTS[variant][grp] = aggregate(smp, outs)
    print(variant, "done")

def results_table(res):
    rows = []
    for grp in EVAL_SETS:
        for metric in ["count_match", "rhyme_hit", "rhyme_consistency", "theme_hit", "copy_rate", "no_latin"]:
            row = {"set": grp, "metric": metric}
            for v in res: row[v] = res[v].get(grp, {}).get(metric)
            if any(row.get(v) is not None for v in res): rows.append(row)
    return pd.DataFrame(rows)

print(results_table(RESULTS).to_string(index=False))
with open(f"{OUT_DIR}/eval_results.json", "w", encoding="utf-8") as f:
    json.dump(RESULTS, f, ensure_ascii=False, indent=1)

BASE done
TUNED done
  set            metric  BASE  TUNED
rhyme       count_match 0.417  1.000
rhyme         rhyme_hit 0.000  0.450
rhyme rhyme_consistency 0.461  0.756
rhyme         theme_hit 1.000  1.000
rhyme         copy_rate 0.000  0.000
rhyme          no_latin 0.983  1.000
theme       count_match 0.200  1.000
theme rhyme_consistency 0.435  0.679
theme         theme_hit 0.625  0.900
theme         copy_rate 0.000  0.000
theme          no_latin 0.950  0.975
plain       count_match 0.467  1.000
plain rhyme_consistency 0.440  0.712
plain         copy_rate 0.000  0.000
plain          no_latin 0.983  1.000


In [ ]:
# 7.3 OPTIONAL: apples-to-apples with the v1 adapter if it is on Drive (v1 never saw rhyme prompts: expect it to fail those)
V1_ADAPTER = f"{WORK_DIR}/Qareed-Qwen2.5-1.5B-Poet-v1/adapter"
if os.path.exists(V1_ADAPTER) and "1.5B" in BASE_MODEL:
    try:
        trainer.model.load_adapter(V1_ADAPTER, adapter_name="v1")
        trainer.model.set_adapter("v1")
        RESULTS["V1"] = {}
        for grp, smp in EVAL_SETS.items():
            if smp:
                outs = gen_many([s["prompt"] for s in smp], trainer.model, system_prompt=V1_SYSTEM_PROMPT)
                RESULTS["V1"][grp] = aggregate(smp, outs)
        print(results_table(RESULTS).to_string(index=False))
        with open(f"{OUT_DIR}/eval_results.json", "w", encoding="utf-8") as f:
            json.dump(RESULTS, f, ensure_ascii=False, indent=1)
    except Exception as e:
        print("v1 comparison skipped:", repr(e)[:300])
    finally:
        try: trainer.model.set_adapter("default")
        except Exception as e: print("could not switch back to the v2 adapter:", e)
else:
    print("v1 adapter not found (or a different base model): skipped")

  set            metric  BASE  TUNED    V1
rhyme       count_match 0.417  1.000 1.000
rhyme         rhyme_hit 0.000  0.450 0.067
rhyme rhyme_consistency 0.461  0.756 0.735
rhyme         theme_hit 1.000  1.000 0.000
rhyme         copy_rate 0.000  0.000 0.000
rhyme          no_latin 0.983  1.000 1.000
theme       count_match 0.200  1.000 1.000
theme rhyme_consistency 0.435  0.679 0.679
theme         theme_hit 0.625  0.900 0.825
theme         copy_rate 0.000  0.000 0.000
theme          no_latin 0.950  0.975 1.000
plain       count_match 0.467  1.000 1.000
plain rhyme_consistency 0.440  0.712 0.665
plain         copy_rate 0.000  0.000 0.000
plain          no_latin 0.983  1.000 0.983


In [ ]:
# 7.4 READ THEM YOURSELF: the first 3 prompts of each set, base vs tuned (no cherry-picking: same seeds, in order)
for grp, smp in EVAL_SETS.items():
    for i, s in enumerate(smp[:3]):
        print("=" * 70); print(f"[{grp}] Q:", s["prompt"])
        print("\n[BASE]\n", SAVED_OUTS["BASE"][grp][i])
        print("\n[TUNED]\n", SAVED_OUTS["TUNED"][grp][i])
        print("\n[REAL]\n", s["answer"])

[rhyme] Q: قل لي ثلاثة أبيات من الشعر على قافية حرف الدال.

[BASE]
 هذا هو الشيء الذي أحببت أن أكتب له:
في عمق الهموم وأنا أتذوق الحب،  
وراء الجهل يسكنون القلوب الصغيرة،  
وألقيت في النيران كالمغفرة.

[TUNED]
 ويا لؤلؤة العين كأنما  ...  عشت في جنة أرضها يد
فكم صرت أستره وأشربها  ...  من خمر الوجد حتى تسد
والحب رزء يشكو كل سكر  ...  ويعللها بعذابه يخد

[REAL]
 ولكم بقلبي من زماني حسرة  ...  لم تشفها عدة الغزال الأغيد
فلئن سلوت فعن فؤاد يائس  ...  نبذ الهوى وغليله لم يبرد
أسوان مستعمر كأن حصاته  ...  تصلى على جمر الغضى المتوقد
[rhyme] Q: انظم بيتين من الشعر، والروي حرف الشين.

[BASE]
 في النجوم الباردة  
الشمس تغطى السحاب الغني  
بسمة خفيفة على وجهها  
تغيب في الليل يدوده.

[TUNED]
 وذا ما كان في القرب من قربه  ...  عجبت أن يعود على اليمين
فكان بالقرب راضيا عنده  ...  فلله لا أرى في الحب مسالم

[REAL]
 طيوره قد تقابلت نسقا  ...  كأنها وقع على العش
فضاؤه طاب فسحة وهوى  ...  مصقل الأرض مؤنق الفرش
[rhyme] Q: انظم بيتًا واحدًا من الشعر على قافية حرف القاف.

[BASE]
 الشمس تشرق في الفجر،  
ت

## 8) Publish (only after you read section 7)

Set `RUN_PUSH=True` in 0.1 (and keep `PUSH_PRIVATE=True` until you are happy). This section works even after a runtime restart (it reloads the adapter from Drive). The merge runs on **CPU** and is saved to local disk, then uploaded; the upload is **verified**.

In [5]:
# 8.1 push the adapter (private by default)
assert RUN_PUSH, "Set RUN_PUSH=True in 0.1 when you want to publish."
assert not SMOKE_TEST, "This is a smoke-test run: nothing to publish."
assert os.path.exists(f"{ADAPTER_DIR}/adapter_config.json"), f"No adapter found in {ADAPTER_DIR}"
api.create_repo(ADAPTER_REPO, private=PUSH_PRIVATE, exist_ok=True)
api.upload_folder(folder_path=ADAPTER_DIR, repo_id=ADAPTER_REPO)
print("https://huggingface.co/" + ADAPTER_REPO)

No files have been modified since last commit. Skipping to prevent empty commit.


https://huggingface.co/Youssefx64/Qareed-Qwen2.5-1.5B-Poet-v2-LoRA


In [7]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [8]:
# 8.2a merge on CPU -> save to LOCAL disk -> sanity checks
assert RUN_PUSH and not SMOKE_TEST, "Set RUN_PUSH=True and SMOKE_TEST=False in 0.1"
import shutil
for name in ["trainer", "model", "pipe", "peft_model", "merged", "base"]:
    if name in globals():
        del globals()[name]
gc.collect(); torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

assert os.path.exists(f"{ADAPTER_DIR}/adapter_config.json"), f"No adapter in {ADAPTER_DIR}"

# fresh tokenizer from the saved adapter folder (the in-memory one has padding_side='left' from section 7)
tok_m = AutoTokenizer.from_pretrained(ADAPTER_DIR)
tok_m.padding_side = "right"

print("1/4 loading base on CPU...")
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16, device_map="cpu", low_cpu_mem_usage=True)
print("2/4 attaching adapter + merging...")
peft_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
merged = peft_model.merge_and_unload()
assert not any("lora" in n.lower() for n, _ in merged.named_parameters()), "LoRA layers are still present after merge"
merged = merged.to(torch.float16)

print("3/4 saving locally...")
shutil.rmtree(MERGED_DIR, ignore_errors=True)
try:
    merged.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size="5GB")
except TypeError as e:
    print("safe_serialization not accepted by this transformers version, saving without it:", e)
    merged.save_pretrained(MERGED_DIR, max_shard_size="5GB")
tok_m.save_pretrained(MERGED_DIR)

print("4/4 checking files...")
files = {f: os.path.getsize(f"{MERGED_DIR}/{f}") for f in sorted(os.listdir(MERGED_DIR))}
for f, sz in files.items():
    print(f"  {f:40s} {sz / 1e6:10.1f} MB")
weights = sum(sz for f, sz in files.items() if f.endswith(".safetensors"))
assert weights > 2.5e9, f"Weights file looks too small ({weights/1e9:.2f} GB): the save is incomplete"
assert "config.json" in files and "generation_config.json" in files, "config files missing"
del merged, peft_model, base; gc.collect()
print("\nOK: merged model saved to", MERGED_DIR)

1/4 loading base on CPU...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

2/4 attaching adapter + merging...
3/4 saving locally...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

4/4 checking files...
  chat_template.jinja                             0.0 MB
  config.json                                     0.0 MB
  generation_config.json                          0.0 MB
  model.safetensors                            3087.5 MB
  tokenizer.json                                 11.4 MB
  tokenizer_config.json                           0.0 MB

OK: merged model saved to /content/Qareed-Qwen2.5-1.5B-Poet-v2-merged


In [10]:
# copy the merged model to Drive (about 3 GB: can take a few minutes)
import shutil
DRIVE_MERGED_DIR = f"{OUT_DIR}/merged"
shutil.rmtree(DRIVE_MERGED_DIR, ignore_errors=True)
shutil.copytree(MERGED_DIR, DRIVE_MERGED_DIR)

for f in sorted(os.listdir(MERGED_DIR)):
    a, b = os.path.getsize(f"{MERGED_DIR}/{f}"), os.path.getsize(f"{DRIVE_MERGED_DIR}/{f}")
    assert a == b, f"copy mismatch: {f} ({a} vs {b})"
    print(f"{f:30s} {b / 1e6:9.1f} MB  OK")
print("\nSaved on Drive:", DRIVE_MERGED_DIR)

chat_template.jinja                  0.0 MB  OK
config.json                          0.0 MB  OK
generation_config.json               0.0 MB  OK
model.safetensors                 3087.5 MB  OK
tokenizer.json                      11.4 MB  OK
tokenizer_config.json                0.0 MB  OK

Saved on Drive: /content/drive/MyDrive/Colab Notebooks/Qareed/Qareed-Qwen2.5-1.5B-Poet-v2/merged


In [9]:
# 8.2b upload the local folder to the Hub and VERIFY it arrived
assert RUN_PUSH and not SMOKE_TEST
assert os.path.exists(f"{MERGED_DIR}/config.json"), "Run 8.2a first"

api.create_repo(MERGED_REPO, private=PUSH_PRIVATE, exist_ok=True)
api.upload_folder(folder_path=MERGED_DIR, repo_id=MERGED_REPO, commit_message="Upload merged fp16 model")

info = api.model_info(MERGED_REPO, files_metadata=True)
sizes = {s.rfilename: s.size for s in info.siblings}
for k, v in sizes.items():
    print(f"  {k:40s} {(v or 0) / 1e6:10.1f} MB")
assert any(k.endswith(".safetensors") and (v or 0) > 2.5e9 for k, v in sizes.items()), \
    "Upload incomplete: no large .safetensors on the Hub. Re-run this cell (it is safe to repeat)."
print("\nVERIFIED:", "https://huggingface.co/" + MERGED_REPO, "| private:", PUSH_PRIVATE)

  .gitattributes                                  0.0 MB
  chat_template.jinja                             0.0 MB
  config.json                                     0.0 MB
  generation_config.json                          0.0 MB
  model.safetensors                            3087.5 MB
  tokenizer.json                                 11.4 MB
  tokenizer_config.json                           0.0 MB

VERIFIED: https://huggingface.co/Youssefx64/Qareed-Qwen2.5-1.5B-Poet-v2 | private: True


In [11]:
# 8.3 model cards (numbers come from section 7; if it was not run, the table is omitted)
assert RUN_PUSH and not SMOKE_TEST
from huggingface_hub import ModelCard
F = "`" * 3

REC = json.load(open(f"{OUT_DIR}/run_record.json", encoding="utf-8")) if os.path.exists(f"{OUT_DIR}/run_record.json") else {}
if "SYSTEM_PROMPT" not in globals():
    SYSTEM_PROMPT = REC.get("system_prompt", "")
n_train = len(train_samples) if "train_samples" in globals() else REC.get("n_train", "?")
if "RESULTS" not in globals():
    p = f"{OUT_DIR}/eval_results.json"
    RESULTS = json.load(open(p, encoding="utf-8")) if os.path.exists(p) else {}

def fmt(x): return "—" if x is None else f"{x:.2f}"
table_md = ""
if RESULTS.get("TUNED"):
    variants = [v for v in ["BASE", "V1", "TUNED"] if v in RESULTS]
    table_md = "| set | metric | " + " | ".join(variants) + " |\n|---|---|" + "---|" * len(variants) + "\n"
    for grp in ["rhyme", "theme", "plain"]:
        for metric in ["count_match", "rhyme_hit", "rhyme_consistency", "theme_hit", "copy_rate", "no_latin"]:
            vals = [RESULTS[v].get(grp, {}).get(metric) for v in variants]
            if any(x is not None for x in vals):
                table_md += f"| {grp} | {metric} | " + " | ".join(fmt(x) for x in vals) + " |\n"
    n_note = f"{RESULTS['TUNED'].get('rhyme', {}).get('n', '?')} prompts per set, sampled at temperature 0.7"
else:
    n_note = ""

card_text = f"""---
language:
- ar
license: apache-2.0
base_model: {BASE_MODEL}
datasets:
- arbml/Ashaar_dataset
library_name: transformers
pipeline_tag: text-generation
tags:
- arabic
- poetry
- qlora
- peft
---

# Qareed Poet ({BASE_MODEL.split('/')[-1].replace('-Instruct', '')}) {VERSION} — قريض

Arabic poetry generation model, QLoRA fine-tuned from `{BASE_MODEL}`. Experimental release.

## Usage
{F}python
from transformers import pipeline
pipe = pipeline("text-generation", model="{MERGED_REPO}", device_map="auto")
SYSTEM = "{SYSTEM_PROMPT}"
msgs = [{{"role": "system", "content": SYSTEM}},
        {{"role": "user", "content": "اكتب بيتين في الشوق على قافية حرف الراء."}}]
print(pipe(msgs, max_new_tokens=120, do_sample=True, temperature=0.8)[0]["generated_text"][-1]["content"])
{F}

Prompts can ask for: a number of baits (1-4), a theme (e.g. الحزن، الغزل، المدح، الرثاء، الشوق، الفراق، العتاب، الهجاء، الدين، حب الوطن), an era, and a rhyme letter ("على قافية حرف الراء").

## Training
- Data: `arbml/Ashaar_dataset` (Ashaar, arXiv:2307.06218), cleaned and deduplicated; colloquial poems removed; poem-level train/validation split with a bait-level leakage check.
- Samples: windows of 1-4 consecutive baits, always single-rhyme; theme prompts only when the window text matches theme keywords (weak, heuristic labels); about half of the prompts request the rhyme letter.
- Method: QLoRA (NF4), LoRA r=32 alpha=64 on all linear layers, loss on the answer only, {n_train} training samples, {EPOCHS} epoch.
- Diacritics stripped. Meter (wazn) is not modeled or controlled.

## Evaluation (automatic heuristics on held-out poems, not human ratings)
{n_note}

{table_md}
`rhyme_hit`: every bait ends with the requested letter (surface rhyme letter; a final alef/maqsura is skipped). `theme_hit`: the output contains a keyword of the theme. `copy_rate`: share of generated baits found verbatim in the training set.

## Limitations
- Meter (wazn) is not controlled or verified; rhyme means the surface rhyme letter only.
- Theme labels come from the source websites and are noisy; the keyword filter is a heuristic.
- The source text lost the letter alef-with-hamza-below; common words were patched but some words may still be wrong.
- May reproduce lines from the training data (see copy_rate). Outputs need human review.
"""
ModelCard(card_text).push_to_hub(MERGED_REPO)
adapter_card = f"""---
base_model: {BASE_MODEL}
library_name: peft
language:
- ar
tags:
- arabic
- poetry
- lora
---

LoRA adapter for `{MERGED_REPO}` (merged full model) — {VERSION}. See that repo for training details, evaluation and limitations.
"""
ModelCard(adapter_card).push_to_hub(ADAPTER_REPO)
print("Model cards pushed.")

Model cards pushed.


## 9) Try the published model (optional)

In [12]:
# 9.1 quick inference test from the local merged copy (or the Hub repo)
from transformers import pipeline
if "SYSTEM_PROMPT" not in globals():
    SYSTEM_PROMPT = json.load(open(f"{OUT_DIR}/run_record.json", encoding="utf-8"))["system_prompt"]
src = MERGED_DIR if os.path.exists(MERGED_DIR) else MERGED_REPO
pipe = pipeline("text-generation", model=src, device_map="auto", dtype=torch.float16)
def ask(q, temperature=0.8):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": q}]
    return pipe(msgs, max_new_tokens=140, do_sample=True, temperature=temperature, top_p=0.9,
                repetition_penalty=1.1)[0]["generated_text"][-1]["content"]
for q in ["اكتب بيتين في الشوق والحنين.", "اكتب بيتين في العتاب على قافية حرف الباء."]:
    print("Q:", q); print(ask(q)); print()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'repetition_penalty', 'temperature', 'max_new_tokens', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Q: اكتب بيتين في الشوق والحنين.


[transformers] Both `max_new_tokens` (=140) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=140) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


يا أبا الهادي يا روح داركم  ...  ما لي لا نعيم وأسى
لا تنامى على جفون عليل  ...  يبكيها حزن وندم

Q: اكتب بيتين في العتاب على قافية حرف الباء.
يا خادم الحسن لو أجبتك طيب  ...  ما جرى بالحق بيني وبينك نقص
وذا كنت يا سيدتي تبكي على  ...  صديق فأنت صديقي ولست عبد



In [13]:
# 9.2 OPTIONAL: temporary public demo link (stops when this session ends). Needs 9.1.
require("ask", hint="Run 9.1 first.")
!pip install -q gradio
import gradio as gr
demo = gr.Interface(fn=lambda q, t: ask(q, float(t)),
    inputs=[gr.Textbox(label="الطلب", value="اكتب بيتين في الشوق على قافية حرف الراء."),
            gr.Slider(0.3, 1.2, value=0.8, step=0.05, label="Temperature")],
    outputs=gr.Textbox(label="الناتج", lines=8),
    title="قريض (Qareed) v2 — تجريبي", description="القافية سطحية والوزن غير مضبوط؛ المخرجات تحتاج مراجعة بشرية.")
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a70c369e916d25876c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Both `max_new_tokens` (=140) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=140) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=140) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a70c369e916d25876c.gradio.live


## Troubleshooting
| Symptom | Fix |
|---|---|
| `MISSING [...]` / `NameError` | The runtime restarted. Re-run from 0.1 (data is reloaded from Drive, so it is quick). |
| `CUDA out of memory` | In 6.1 set `BS, ACC = 4, 4`, restart the runtime, rerun. |
| Cell 6.2 raises "loss still covers the prompt" | Do not train. Send me the printed output (TRL API changed). |
| `TypeError` in `SFTConfig` | A TRL argument was renamed: send me the message. |
| Colab disconnected during training | Restart, run 0.1 -> 0.4, 1.1 -> 5.1 (data loads from Drive), 6.1, 6.2, 6.3: it resumes from the last checkpoint. |
| Version mismatch warning in 0.3 | Re-run 0.2, restart the runtime, continue from 0.3. |
| Evaluation too slow | Lower `EVAL_N` in 7.1. |